# Colab launcher

Runs the GPU work on Colab. This notebook holds no project logic: it prepares the VM and calls the
same commands as on a local machine, with `GEO_OVERLAYS=configs/colab.yaml` so every command uses
the Colab paths (data and caches on the VM's local disk, run outputs on Drive).

**Runtime → Change runtime type:** A100 for full fine-tunes; L4 is enough for the smoke runs and the
probe. Avoid T4 (no bfloat16) and TPUs.

**Colab secrets (key icon), notebook access on:** `GH_TOKEN` (GitHub, read, while the repo is
private) and `HF_TOKEN` (Hugging Face; DINOv3 access must be granted on the hub).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/geo-relations"
!mkdir -p {DRIVE}/runs {DRIVE}/data_tarballs

In [ ]:
import os
from google.colab import userdata

REPO = "parsa-hke/veridiagram"  # GitHub owner/name
BRANCH = "main"
WORKDIR = "/content/veridiagram"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GH_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} pull --ff-only
else:
    !git clone -q -b {BRANCH} {url} {WORKDIR}
    !git -C {WORKDIR} remote set-url origin https://github.com/{REPO}.git  # drop token from .git/config
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# Pinned versions (numpy/pandas match Colab's own packages). If pip asks for a restart, use
# Runtime -> Restart session, rerun the cells above (idempotent) and continue.
!pip install -q -r requirements.txt

In [ ]:
os.environ["HF_HOME"] = "/content/hf_cache"  # local disk: safetensors are memory-mapped, which the Drive mount does not support reliably
os.environ["GEO_OVERLAYS"] = "configs/colab.yaml"       # every CLI uses the Colab paths
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
!python -m experiments.run --device-info

## Data

First session: downloads the datasets, builds everything (`python -m data.prepare`, ~15-20 min) and
saves one tarball to Drive. Later sessions: unpack that tarball to the local disk (~2 min). The
tarball name carries the git hash of the data pipeline, so a pipeline change rebuilds automatically.

In [ ]:
# The tarball is named after data/VERSION (bumped only when the data pipeline's outputs change).
VERSION = open("data/VERSION").read().split()[0]
tarball = f"{DRIVE}/data_tarballs/data_v{VERSION}.tar"
legacy = f"{DRIVE}/data_tarballs/data_4fa8896.tar"          # first tarball, identical to v1
if not os.path.exists(tarball) and VERSION == "1" and os.path.exists(legacy):
    os.rename(legacy, tarball)
print("data version", VERSION, "->", tarball)
if os.path.exists(tarball):
    !mkdir -p /content/data && tar -C /content -xf {tarball}
else:
    !python -m data.prepare --download
    !tar -C /content -cf {tarball} data
!du -sh /content/data/raw /content/data/processed

## Phase 6 gate: smoke runs (must pass before any full run)

In [ ]:
# DINOv3 access check: loads the gated checkpoint with NO fallback; must end with "OK".
!python -m models.backbone

In [ ]:
!python -m experiments.run --config configs/probe.yaml --smoke
!python -m experiments.run --config configs/finetune.yaml --smoke
!python -m experiments.gate smoke

## Part B teacher (from the lab)

The 7.3 model trained in the lab (`finetune_synth10k_s2`, best val macro-F1 of the three seeds) is
the self-training teacher. Its checkpoint is unpacked once from the lab archive on Drive
(`geo-relations/lab_results/geo_ckpt_synth10k.tar`) to `runs/finetune_synth10k_s2/checkpoints/best.pt`
and checked against its sha256.

In [ ]:
import hashlib, tarfile
TEACHER = f"{DRIVE}/runs/finetune_synth10k_s2/checkpoints/best.pt"
TEACHER_SHA256 = "07db988c20212b7275ab38316d22890271c4fb6c76070aa5a98d1a1d994d0bb9"   # verified on the laptop
ARCHIVE = f"{DRIVE}/lab_results/geo_ckpt_synth10k.tar"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

if not os.path.exists(TEACHER):
    with tarfile.open(ARCHIVE) as t:
        t.extract("runs/finetune_synth10k_s2/checkpoints/best.pt", path=DRIVE, filter="data")
ok = sha256(TEACHER) == TEACHER_SHA256
print("teacher", TEACHER, os.path.getsize(TEACHER), "bytes, sha256", "OK" if ok else "MISMATCH")
assert ok, "teacher checkpoint differs from the lab copy: delete it and rerun this cell"

## Phase 7 gate: smoke runs of the new modes (must pass before the queue)

The no-overlay ablation (7.8) and self-training (Part B, with the real teacher) on the smoke subset,
then the gate check. About 5 minutes.

In [ ]:
!python -m experiments.run --config configs/ablation_nooverlay.yaml --smoke
!python -m experiments.run --config configs/selftrain_consistency.yaml --smoke
!python -m experiments.gate phase7

## Full runs

`configs/queue_phase7_colab.yaml`: the 13 remaining Phase 7 runs (Part A 1k and 50k, Part B, 7.5 and
7.8), seed 0 of every experiment first. Results go to `MyDrive/geo-relations/runs/<name>_s<seed>/` and
`runs/log.csv`. If the session disconnects, run every cell again from the top: finished runs are
skipped and an interrupted fine-tune resumes from its last epoch.

In [ ]:
RUN_FULL = False   # set to True only after both gates above printed PASS ("Run all" never starts full runs)
QUEUE = "configs/queue_phase7_colab.yaml"
!python -m experiments.queue {QUEUE} --status
if RUN_FULL:
    !python -m experiments.queue {QUEUE}
else:
    print("RUN_FULL is False: skipping", QUEUE)

## Pack the results for the laptop

Metrics, predictions and histories of every Colab run (no checkpoints), plus the Part B teacher
predictions, into one small file on Drive: `geo-relations/geo_results_colab.tar.gz`. Download it to
the laptop when the queue has finished.

In [ ]:
!python -m experiments.queue {QUEUE} --status
!cd {DRIVE} && tar --exclude='*/checkpoints' --exclude='selftrain_teacher/*_smoke' -czf geo_results_colab.tar.gz runs && ls -lh geo_results_colab.tar.gz